# 4일차 실습 ② — ONNX Runtime 으로 추론하기

부경대학교 교내 컴퓨터비전 부트캠프 · 2026. 8. 6. · 3차시 (11:30 – 12:20)

---

지금까지는 `model(image)` 한 줄이면 결과가 나왔습니다. 그 한 줄 안에서 무슨 일이
벌어지고 있었는지를 직접 짜 보는 시간입니다.

| STEP | 하는 일 |
|---|---|
| 0 · 1 | 환경 준비, yolo11n 을 .onnx 로 내보내기 |
| 2 | 세션을 열고 입력·출력 모양 확인 |
| 3 | 전처리 직접 짜기 (**TODO 1**) |
| 4 | 출력 텐서에서 박스·클래스·점수 꺼내기 (**TODO 2**) |
| 5 | conf 로 거르고 NMS 적용 (**TODO 3**) |
| 6 | 속도 비교 · ultralytics 결과와 대조 (**TODO 4**) |

## STEP 0 · 환경 준비

In [ ]:
!pip install -q ultralytics onnxruntime onnxslim

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110


def show(img_bgr, title=None, w=10):
    h = w * img_bgr.shape[0] / img_bgr.shape[1]
    plt.figure(figsize=(w, h))
    plt.imshow(img_bgr[:, :, ::-1])          # BGR -> RGB
    plt.axis("off")
    if title:
        plt.title(title, fontsize=13)
    plt.show()


def overlay(img_bgr, mask, color=(60, 120, 240), alpha=0.55):
    """마스크를 원본 위에 반투명하게 얹는다. mask 는 (H, W) bool 배열."""
    out = img_bgr.copy()
    layer = np.zeros_like(out)
    layer[mask.astype(bool)] = color
    return cv2.addWeighted(out, 1.0, layer, alpha, 0)


import time
print("준비 완료")

## STEP 1 · ONNX 로 내보내기

`export()` 한 줄입니다. 몇 초 걸립니다.

- `imgsz=640` — 입력 해상도를 **고정**합니다. 이 값이 나중에 전처리와 맞아야 합니다.
- `simplify=True` — 그래프에서 불필요한 연산을 정리합니다.
- `opset=13` — 대부분의 런타임이 지원하는 안전한 버전입니다.

In [ ]:
from ultralytics import YOLO
from ultralytics.utils import ASSETS
import os

IMG = str(ASSETS / "bus.jpg")

model = YOLO("yolo11n.pt")
onnx_path = model.export(format="onnx", imgsz=640, simplify=True, opset=13)

print("내보낸 파일 :", onnx_path)
print("pt   크기   : %.1f MB" % (os.path.getsize("yolo11n.pt") / 1e6))
print("onnx 크기   : %.1f MB" % (os.path.getsize(onnx_path) / 1e6))

파일이 오히려 커졌습니다. ONNX 는 가중치를 압축하지 않고 그래프 구조까지 함께 담기
때문입니다. **크기가 아니라 '어디서든 돈다' 가 ONNX 의 이점**입니다.

## STEP 2 · 세션 열기

`InferenceSession` 이 ONNX Runtime 의 시작점입니다. 무엇을 넣고 무엇이 나오는지부터
확인하는 것이 순서입니다.

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

inp = session.get_inputs()[0]
out = session.get_outputs()[0]

print("입력 :", inp.name, inp.shape, inp.type)
print("출력 :", out.name, out.shape, out.type)
print()
print("사용 가능한 provider:", ort.get_available_providers())

```
입력 : images  [1, 3, 640, 640]  tensor(float)
출력 : output0 [1, 84, 8400]     tensor(float)
```

이 두 줄이 전부입니다. 우리가 해야 할 일은

1. 이미지를 `(1, 3, 640, 640)` float32 로 만들어 넣고
2. 나온 `(1, 84, 8400)` 을 박스와 클래스로 해석하는 것

## STEP 3 · 전처리

네 가지를 맞춰야 합니다.

| # | 무엇을 | 어떻게 |
|---|---|---|
| ① | 크기 | `cv2.resize(img, (640, 640))` |
| ② | 채널 순서 | BGR → RGB — `[:, :, ::-1]` |
| ③ | 축 순서 | HWC → CHW — `.transpose(2, 0, 1)` |
| ④ | 배치·자료형·범위 | `[None]`, `float32`, `/ 255.0` |

### TODO 1

In [ ]:
def preprocess(img_bgr, size=640):
    # TODO 1 ── 네 줄을 채우세요
    x = ...          # ① 크기를 (size, size) 로
    x = ...          # ② BGR -> RGB
    x = ...          # ③ HWC -> CHW
    x = ...          # ④ 배치 차원 추가 · float32 · 255 로 나누기
    return np.ascontiguousarray(x)


img = cv2.imread(IMG)
x = preprocess(img)
print(x.shape, x.dtype, x.min(), x.max())       # (1, 3, 640, 640) float32 0.0 1.0

제대로 됐다면 `(1, 3, 640, 640) float32 0.0 1.0` 이 찍힙니다.
**하나라도 틀리면 오류 없이 이상한 결과만 나옵니다** — 배포에서 가장 흔한 사고입니다.

이제 실행해 봅니다. (그대로 실행)

In [ ]:
y = session.run(None, {inp.name: x})[0]
print("출력 모양:", y.shape)          # (1, 84, 8400)
print("값의 범위: %.1f ~ %.1f" % (y.min(), y.max()))

## STEP 4 · 출력 텐서 읽기

`(1, 84, 8400)` 의 의미

- `8400` — 모델이 내놓은 **후보 위치의 개수**
- `84` — 후보 하나당 붙는 값. 앞 `4` 개가 박스 `cx, cy, w, h`, 뒤 `80` 개가 클래스 점수

후보 하나가 한 줄이 되도록 **전치**한 뒤, 뒤쪽 80개에서 가장 큰 값과 그 위치를 꺼냅니다.

### TODO 2

In [ ]:
y = session.run(None, {inp.name: x})[0][0]   # (84, 8400)
y = ...                                       # TODO 2 ── (8400, 84) 로 전치

boxes  = ...                                  # 앞 4개 — cx, cy, w, h
scores = ...                                  # 뒤 80개 — 클래스 점수
cls    = ...                                  # 가장 큰 값의 위치
conf   = ...                                  # 가장 큰 값

print("후보 수:", len(boxes))
print("conf 최대: %.3f  최소: %.5f" % (conf.max(), conf.min()))

8400개 중 대부분은 confidence 가 아주 낮습니다. 분포를 보면 왜 걸러 내야 하는지 분명해집니다.
(그대로 실행)

In [ ]:
plt.figure(figsize=(9, 3.2))
plt.hist(conf, bins=60, color="#156082")
plt.yscale("log")
plt.axvline(0.25, color="#E97132", lw=2, ls="--")
plt.text(0.27, plt.ylim()[1] * 0.3, "conf = 0.25", color="#E97132", fontsize=11)
plt.xlabel("confidence"); plt.ylabel("후보 개수 (로그 눈금)")
plt.tight_layout(); plt.show()

print("conf >= 0.25 인 후보:", int((conf >= 0.25).sum()), "/", len(conf))

## STEP 5 · 후처리 — conf 필터와 NMS

어제 슬라이드에서 코드로 따라갔던 그 과정입니다. 이번에는 실제로 부릅니다.

좌표를 원본 크기로 되돌리는 것도 여기서 합니다 — 640×640 기준으로 나온 값이므로
가로세로 배율을 각각 곱해 줍니다.

### TODO 3

In [ ]:
H, W = img.shape[:2]
sx, sy = W / 640, H / 640

keep = ...                                     # TODO 3-① conf 0.25 이상만
b, c, k = boxes[keep], conf[keep], cls[keep]
print("conf 필터 후:", len(b))

# cxcywh(640) -> 좌상단 기준 xywh(원본)
x1 = (b[:, 0] - b[:, 2] / 2) * sx
y1 = (b[:, 1] - b[:, 3] / 2) * sy
bw = b[:, 2] * sx
bh = b[:, 3] * sy
rects = np.stack([x1, y1, bw, bh], axis=1)

idx = ...                                      # TODO 3-② cv2.dnn.NMSBoxes(rects, conf, 0.25, 0.7)
idx = np.array(idx).flatten()
print("NMS 후:", len(idx))

8400 → 수십 개 → 몇 개. 이제 그려 봅니다. (그대로 실행)

In [ ]:
names = model.names
o = img.copy()
for i in idx:
    X1, Y1 = int(x1[i]), int(y1[i])
    X2, Y2 = int(x1[i] + bw[i]), int(y1[i] + bh[i])
    label = f"{names[int(k[i])]} {c[i]:.2f}"
    cv2.rectangle(o, (X1, Y1), (X2, Y2), (232, 96, 21), 3)
    cv2.putText(o, label, (X1, Y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (232, 96, 21), 2)
    print(label, (X1, Y1, X2, Y2))

show(o, "ONNX Runtime + 직접 짠 후처리")

## STEP 6 · 대조와 속도 비교

### TODO 4 — 원래 결과와 맞춰 보기

`YOLO("yolo11n.pt")` 로 같은 이미지를 추론해 개수와 좌표를 비교하세요.

In [ ]:
# TODO 4 ── ultralytics 로 같은 이미지를 추론해 비교하세요
ref = ...                    # model(IMG, conf=0.25, verbose=False)[0]
print("ultralytics(.pt) :", len(ref.boxes), "개")
print("직접 짠 파이프라인 :", len(idx), "개")
print()
for box in ref.boxes:
    xx1, yy1, xx2, yy2 = box.xyxy[0].tolist()
    print(f"{ref.names[int(box.cls)]:<10} {float(box.conf):.2f}  "
          f"({xx1:.0f}, {yy1:.0f}) - ({xx2:.0f}, {yy2:.0f})")

**개수는 같은데 좌표와 점수가 조금씩 다를 것입니다.** 버그가 아닙니다.

우리는 `cv2.resize` 로 640×640 에 **찌그러뜨려** 넣었지만, ultralytics 는 **letterbox** —
가로세로 비율을 지키고 남는 곳을 회색으로 채우는 방식 — 을 씁니다. 모델은 그대로인데
입력이 달라진 것이죠.

**배포에서 성능이 떨어지는 가장 흔한 원인이 바로 이것입니다.** 변환한 모델을 검증할 때는
반드시 같은 이미지로 출력을 대조해 보세요.

ultralytics 로 .onnx 를 직접 열면 letterbox 까지 그대로 써 주므로 결과가 일치합니다.
(그대로 실행)

In [ ]:
onnx_model = YOLO(onnx_path)
r_onnx = onnx_model(IMG, conf=0.25, verbose=False)[0]
print("ultralytics(.onnx):", len(r_onnx.boxes), "개")
for box in r_onnx.boxes:
    print(f"  {r_onnx.names[int(box.cls)]:<10} {float(box.conf):.2f}")

### 속도 비교 (그대로 실행)

In [ ]:
def bench(fn, n=10):
    for _ in range(3):
        fn()
    t = time.time()
    for _ in range(n):
        fn()
    return (time.time() - t) / n * 1000


t_pt  = bench(lambda: model(img, verbose=False))
t_ort = bench(lambda: session.run(None, {inp.name: x}))

print(f"PyTorch  (전처리+추론+후처리) : {t_pt:6.1f} ms")
print(f"ONNX RT  (모델 실행만)        : {t_ort:6.1f} ms")
print(f"                       배율   : {t_pt / t_ort:.1f}x")

공정한 비교는 아닙니다 — PyTorch 쪽은 전처리·후처리까지 포함한 시간이니까요.
그래도 **모델 실행 자체가 얼마나 빨라지는지**는 볼 수 있습니다.

> Colab GPU 런타임이라면 PyTorch 가 GPU 를, ONNX Runtime 이 CPU 를 쓰고 있어
> 오히려 PyTorch 가 빠르게 나옵니다. 그럴 때는 `런타임 유형 → CPU` 로 바꿔
> 다시 재 보면 ONNX 쪽이 앞섭니다. **ONNX 의 이점은 GPU 가 없는 환경에서 드러납니다.**

---

## 정리

1. 전처리에서 맞춰야 하는 네 가지는 … 이다
2. `(1, 84, 8400)` 에서 84 는 … 이고 8400 은 … 이다
3. 우리 결과와 ultralytics 결과가 달랐던 이유는 … 이다

### 다음 차시

4차시에는 최종 프로젝트 주제를 고릅니다. 오늘 만든 것들이 그대로 출발점이 됩니다.

- 주제 1 — 앞 노트북의 **YOLO + SAM** 파이프라인
- 주제 2 — 앞 노트북의 **mask_iou** 함수
- 주제 3 — 어제의 **threshold 실험**